# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suha-2004/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: Can historical search and content signals be used to identify pages that should be prioritized for content review?

Decision supported: This analysis helps editors decide which pages should be reviewed or refreshed first.

Unit of analysis: Page.

Output: A ranked content-opportunity score with reasons for prioritization.

In [1]:
# Capstone setup

import os
import duckdb
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt

print("Setup complete")

Setup complete


In [2]:
# Connect to the FlyRank full release

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it to Colab Secrets.")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("FlyRank warehouse connected")

FlyRank warehouse connected


In [3]:
date_range = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
""").df()

date_range

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date
0,2025-01-27,2026-06-30


In [4]:
clients = con.sql(f"""
    SELECT
        client_hash_id,
        access_profile,
        gsc_data_start,
        ga4_data_start
    FROM {TABLES['dim_clients']}
""").df()

print("Number of clients:", len(clients))
clients.head()

Number of clients: 104


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,gsc_and_ga4,NaT,2026-05-22
1,client_05475c07ed21a83a,no_search_or_analytics_access,NaT,NaT
2,client_06d356715a8ff3b6,gsc_and_ga4,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,no_search_or_analytics_access,2025-11-05,NaT
4,client_08a6a72ff48e62c0,gsc_only,2025-09-24,NaT


In [5]:
bounds = con.sql(f"""
    SELECT MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
""").df()

end_date = bounds.loc[0, "end_date"]

print("Dataset end date:", end_date)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset end date: 2026-06-30 00:00:00


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),

    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_prev30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_prev30

        FROM {TABLES['fact_daily']} f
        CROSS JOIN bounds b

        WHERE f.report_date > b.end_d - INTERVAL 60 DAY

        GROUP BY
            f.client_hash_id,
            f.content_hash_id

        HAVING imp_prev30 >= 100
    )

    SELECT *
    FROM windowed
""").df()

print("Feature rows:", len(features))
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
